In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import os

# 1. 크롬 옵션 설정 (다운로드 자동화)
chrome_options = webdriver.ChromeOptions()
prefs = {"download.default_directory": os.getcwd() + "\\downloads"}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=chrome_options)
driver.get("https://www.opinet.co.kr/user/cufaq/cufaqSelect.do")
time.sleep(3)

data_list = []

# 2. 1~4페이지 순회
for page in range(1, 5):
    print(f"--- {page}페이지 수집 시작 ---")
    driver.execute_script(f"fn_egov_link_page({page})")
    time.sleep(2)
    
    # [핵심] 페이지가 바뀔 때마다 목록 페이지의 주소를 저장
    list_url = driver.current_url 
    
    # 현재 페이지의 게시물 개수 파악
    count = len(driver.find_elements(By.CSS_SELECTOR, "td.t_left.input a"))
    
    for i in range(count):
        # 매번 목록 요소를 다시 찾아서 클릭 (IndexError 방지)
        titles = driver.find_elements(By.CSS_SELECTOR, "td.t_left.input a")
        title_text = titles[i].text
        titles[i].click()
        time.sleep(1.5)
        
        # 상세 내용 추출
        content = driver.find_element(By.CLASS_NAME, "view_contents").text
        
        # 첨부파일 다운로드
        try:
            file_link = driver.find_element(By.CSS_SELECTOR, ".file li a")
            file_link.click()
            print(f"  -> 파일 다운로드: {file_link.text}")
            time.sleep(2)
        except:
            print("  -> 첨부파일 없음")
            
        data_list.append({"제목": title_text, "내용": content})
        
        # [핵심] 목록 페이지로 강제 이동 (driver.back() 대신 사용)
        driver.get(list_url)
        time.sleep(2)

# 결과 저장
df = pd.DataFrame(data_list)
df.to_excel("final_faq_data.xlsx", index=False)
driver.quit()
print("모든 작업이 완료되었습니다!")